# Fase 2 — Post-filtro (barrido) y comparación con mono

Sobre el algoritmo ganador de la Fase 1 (**NM-MVDR**, sin WPE) se estudian las
variantes de **post-filtro** y se compara contra el **DTLN monocanal** (baseline
single-channel automático). Escenario acústicamente diverso; el foco es el barrido
**RT60 × SNR** para ver tendencias y **diseñar un post-filtro adaptativo** (más
agresivo a SNR bajo).

- **Beamformer único:** NM-MVDR (el de la Fase 1). No se comparan otros BF acá.
- **Post-filtros (PLACEHOLDER):** varias variantes/hiperparámetros; muchas **aún
  sin diseñar**. Acá van como barrido de `smooth` sobre `PF`/`BANPF`; agregá las
  nuevas a medida que se definan.
- **Baseline mono:** DTLN-mono sale automático si hay interpreters cargados.

*Salida:* elegir/diseñar el post-filtro óptimo con parámetros automáticos según el
entorno (RT/SNR).

**Correr en orden:** Setup → Config+Run → Análisis → Figuras.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os, importlib, importlib.util

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive
ip = get_ipython()

# (modulo que se importa, spec para instalar). git+ para las libs de GitHub;
# el resto por nombre. Mismos specs de siempre -> NO fuerza rebuild del cache.
PKGS = [
    ("noisereduce",     "noisereduce"),
    ("mir_eval",        "mir_eval"),
    ("pystoi",          "pystoi"),
    ("pesq",            "pesq"),
    ("paderbox",        "paderbox"),
    ("ai_edge_litert",  "ai_edge_litert"),
    ("pb_bss",          "git+https://github.com/fgnt/pb_bss.git"),
    ("pyroomacoustics", "git+https://github.com/LCAV/pyroomacoustics.git"),
    ("nara_wpe",        "git+https://github.com/fgnt/nara_wpe.git"),
    ("fast_bss_eval",   "git+https://github.com/fakufaku/fast_bss_eval.git"),
]
def _name(spec):  # nombre instalable desde el cache (sin git+/.git)
    return spec.rsplit("/", 1)[-1].replace(".git", "") if spec.startswith("git+") else spec
BUILD = [spec for _, spec in PKGS]

# 1) (Re)construir el cache de wheels en Drive SOLO si cambio la lista (manifest).
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
if (not os.path.isfile(manifest)) or open(manifest).read() != key:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(BUILD))
    with open(manifest, "w") as fh:
        fh.write(key)

# 2) Instalar desde el cache PAQUETE POR PAQUETE. Si a uno le falta la wheel
#    (p.ej. Colab cambio de version de Python), NO bloquea a los demas.
for mod, spec in PKGS:
    if importlib.util.find_spec(mod) is None:
        ip.system(f"pip install --no-index --find-links={WHL} {_name(spec)}")

# 3) AUTOCURA: lo que SIGA sin poder importarse se instala desde el indice
#    (PyPI/git) y se agrega al cache para la proxima sesion. Esto arregla el
#    'ModuleNotFoundError: pyroomacoustics' cuando la wheel cacheada no sirve.
importlib.invalidate_caches()
missing = [(m, s) for m, s in PKGS if importlib.util.find_spec(m) is None]
if missing:
    print("[!] Faltan tras el cache:", [m for m, _ in missing], "-> instalando desde el indice...")
    ip.system("pip install " + " ".join(s for _, s in missing))
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(s for _, s in missing))
    importlib.invalidate_caches()

# 4) Verificacion final.
faltan = [m for m, _ in PKGS if importlib.util.find_spec(m) is None]
if faltan:
    print(f"[!] SIGUEN faltando {faltan}: reinicia el runtime "
          f"(Entorno de ejecucion > Reiniciar) y reejecuta esta celda.")
else:
    print("[*] Todas las dependencias OK.")
# Nuclear (si el cache quedo inservible tras un cambio de Python de Colab):
#   !rm -rf /content/drive/MyDrive/colab_wheels   y reejecuta -> reconstruye todo.

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config + Ejecución (Fase 2)

In [ ]:
import sys, os, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import (NM_MVDR, DTLN_MB_MVDR_SOUDEN_BAN,
                                    NM_MVDR_PF, NM_MVDR_BAN_PF)
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK.")
else:
    print("[!] Sin DTLN interpreters.")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15
SMOOTHS  = [0.2, 0.33, 0.5, 0.66]   # <<< PLACEHOLDER: hiperparámetros del post-filtro
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav", "hairdryer_07_SH_MKH800.wav", "drill_07_RHODE_NT1.wav",
]]
# Estrés espacial FIJO (no es el foco): 1 interferente y 3 simultáneos.
INTERF_CONFIGS = [
    [(45, 1.0, 0)],
    [(45, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],
]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.050,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}

# --- GRILLA: barrido comprensivo RT60 x SNR (ejes primarios) ---
param_grid = {
    'rt60':           [0.160, 0.360, 0.610],
    'isir_db':        [-5, 0, 5, 10, 15],     # denso: ver la tendencia por SNR
    'target_angle':   [0], 'target_dist': [1.0],
    'source_path':    TARGETS,
    'interf_configs': INTERF_CONFIGS,
    'use_wpe':        [False],
    'mismatch_gain':  [0], 'mismatch_phase': [0],
    'error_angle_deg':[0.0], 'error_distance_m':[0.0],
}

# --- PROCESADORES (Fase 2: SOLO NM-MVDR + variantes de post-filtro) ---
# PLACEHOLDER: varias variantes no están diseñadas todavía. Todas comparten el MISMO
# beamformer (NM-MVDR); cambia sólo el post-filtro. DTLN-mono sale automático.
processors_dict = {
    "NM-MVDR": NM_MVDR(min_loading=1e-6, alpha=0.99),        # base sin post-filtro
    "BAN":     DTLN_MB_MVDR_SOUDEN_BAN(min_loading=1e-6),    # normalización analítica
}
for s in SMOOTHS:
    processors_dict[f"PF_{s}"]    = NM_MVDR_PF(min_loading=1e-6, alpha=0.99, smooth=s)
    processors_dict[f"BANPF_{s}"] = NM_MVDR_BAN_PF(min_loading=1e-6, alpha=0.99, smooth=s)
# TODO: agregar acá las variantes NUEVAS de post-filtro a medida que se diseñen.

n_cells = 3*len(TARGETS)*len(INTERF_CONFIGS)*len(param_grid['isir_db'])
print("="*60)
print(f"FASE 2 | celdas={n_cells} x {len(processors_dict)} proc")
print("procesadores:", list(processors_dict.keys()))
print("="*60)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp_dir  = f"/content/results_temp/F2_postfiltro_{RUN_TAG}"
drive_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results/F2_postfiltro_{RUN_TAG}"
os.makedirs(temp_dir, exist_ok=True); os.makedirs(drive_dir, exist_ok=True)

df_F2 = run_mird_grid_search(
    grid_params=param_grid, dataset_provider=provider, processors=processors_dict,
    scene_base_config=base_config, output_dir=temp_dir,
    interpreter_1=interpreter_1, interpreter_2=interpreter_2,
    save_catalog=False, apply_dtln_post=False)
shutil.copytree(temp_dir, drive_dir, dirs_exist_ok=True)
print(f"[EXITO] Fase 2 guardada en {drive_dir}")

### Preview rápido (en memoria)

In [ ]:
# Δ PESQ vs iSIR por post-filtro (tendencia para el diseño adaptativo). Usa df_F2.
import matplotlib.pyplot as plt
def _famc(n):
    return "tab:blue" if n.startswith("BANPF") else ("tab:red" if n.startswith("PF_") else "gray")
fig, ax = plt.subplots(figsize=(9, 4))
for name in df_F2.processor.unique():
    g = df_F2[df_F2.processor == name].groupby("isir_db")["Delta_tot_PESQ_early"].mean()
    ax.plot(g.index.values, g.values, "-o", ms=4, color=_famc(name), alpha=0.85, label=name)
ax.set_xlabel("iSIR [dB]"); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2); ax.set_title("Preview Fase 2 — Δ PESQ vs iSIR por post-filtro")
plt.tight_layout(); plt.show()

## Análisis — trade-off del post-filtro y tendencias RT/SNR

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(os.path.join(drive_dir, "mird_benchmark_metrics.csv"))

MET  = [("Delta_tot_PESQ_early","PESQ"), ("Delta_tot_STOI_early","STOI"),
        ("Delta_tot_SDR_early","SDR"), ("Delta_tot_SIR_early","SIR"),
        ("Delta_tot_SAR_early","SAR")]
cols = [c for c,_ in MET if c in df.columns]

print("=== Δ end-to-end (media sobre escenas) por procesador -> trade-off ===\n")
tab = df.groupby("processor")[cols].mean().rename(columns=dict(MET)).round(3)
print(tab.sort_values("PESQ", ascending=False).to_string())

# tendencia por SNR: qué post-filtro gana a cada iSIR (para el diseño adaptativo)
print("\n=== Δ PESQ medio por (procesador x iSIR) -> ¿más agresivo a SNR bajo? ===")
piv = df.pivot_table(index="processor", columns="isir_db",
                     values="Delta_tot_PESQ_early", aggfunc="mean").round(3)
print(piv.to_string())

## Figuras — trade-off y tendencia por SNR/RT (diseño adaptativo)

In [ ]:
import matplotlib.pyplot as plt
df  = pd.read_csv(os.path.join(drive_dir, "mird_benchmark_metrics.csv"))
agg = df.groupby("processor")[[c for c,_ in MET if c in df.columns]].mean()

def fam_color(name):
    if name.startswith("BANPF"): return "tab:blue"
    if name.startswith("PF_"):   return "tab:red"
    return "gray"

# --- Fig A: trade-off. x = Δ STOI / Δ SAR ; y = Δ PESQ. Cada punto un post-filtro. ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, xcol, xlab in [(axes[0],"Delta_tot_STOI_early","Δ STOI"),
                       (axes[1],"Delta_tot_SAR_early","Δ SAR")]:
    if xcol not in agg.columns: continue
    for name, r in agg.iterrows():
        ax.scatter(r[xcol], r["Delta_tot_PESQ_early"], c=fam_color(name), s=60, zorder=3)
        ax.annotate(name, (r[xcol], r["Delta_tot_PESQ_early"]), fontsize=7,
                    xytext=(4,4), textcoords="offset points")
    ax.set_xlabel(xlab); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3)
fig.suptitle("Fase 2 — Trade-off del post-filtro (elegir en la frontera)")
fig.tight_layout(); fig.savefig(os.path.join(drive_dir,"F2_tradeoff.png"), dpi=140, bbox_inches="tight"); plt.show()

# --- Fig B: Δ PESQ vs iSIR por post-filtro (justifica el post-filtro ADAPTATIVO) ---
fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, lbl in [(axes[0],"Delta_tot_PESQ_early","Δ PESQ"),
                     (axes[1],"Delta_tot_SIR_early","Δ SIR [dB]")]:
    for name in agg.index:
        g = df[df.processor==name].groupby("isir_db")[col].mean()
        ax.plot(g.index.values, g.values, "-o", ms=4, color=fam_color(name), alpha=0.8, label=name)
    ax.set_xlabel("iSIR [dB]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=2)
fig2.suptitle("Fase 2 — Tendencia por SNR (más agresivo conviene a iSIR bajo)")
fig2.tight_layout(); fig2.savefig(os.path.join(drive_dir,"F2_vs_isir.png"), dpi=140, bbox_inches="tight"); plt.show()

# --- Fig C: Δ PESQ vs RT60 por post-filtro ---
fig3, ax = plt.subplots(figsize=(8, 5))
for name in agg.index:
    g = df[df.processor==name].groupby("rt60")["Delta_tot_PESQ_early"].mean()
    ax.plot(g.index.values*1000, g.values, "-o", ms=4, color=fam_color(name), alpha=0.8, label=name)
ax.set_xlabel("RT60 [ms]"); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
ax.set_title("Fase 2 — Tendencia por RT60")
fig3.tight_layout(); fig3.savefig(os.path.join(drive_dir,"F2_vs_rt.png"), dpi=140, bbox_inches="tight"); plt.show()